# M0 · Setup check

**Goal:** make sure Kaggle can get our code from GitHub and sees the GPU.

**Kaggle settings (right-hand panel):** Accelerator = **GPU T4 x1**, Internet = **On**.

Workflow for the whole project:
1. Edit code on your laptop → `git commit` → `git push`.
2. On Kaggle, run the setup cell below. It pulls the newest code.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

## Is the GPU there?

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name())
    print("GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## Run the unit tests

The tests check the cache simulator on tiny hand-made examples, and check the
offloading code on a tiny random OLMoE. Everything should say **passed**.

In [ ]:
!python -m pytest -q tests

## Project settings (from `config.py`)

In [ ]:
import config
for name in ["MOE_MODEL", "DENSE_SMALL", "DENSE_LARGE", "NUM_LAYERS", "NUM_EXPERTS", "TOP_K",
             "TOKENS_PER_DATASET", "CACHE_SIZES", "OUTPUT_DIR"]:
    print(f"{name:20s} = {getattr(config, name)}")